# LEAP-HI Extreme Weather Travel Behavior — Data Preparation & Ordered Probit Models

This notebook is a rebuilt, end-to-end version of `Model_Estimation.ipynb`. Running it top to bottom will:

1. Load the raw survey CSV and fix two data-quality issues found in it (`hispanic`, `ac`)
2. Build the derived variables used across all models (severity-level dummies, combined categorical variables)
3. Export **five event-level data files** (Heat, Cold, Power Outage, Earthquake, Flooding), each containing only that event's own rows and variables
4. Fit **ordered probit models** (baseline + backward-elimination final model) for every Event x Activity outcome, including **Power Outage**, which the original notebook didn't yet cover
5. Export a consolidated **Model_Results.xlsx** and a detailed **Codebook.xlsx**

**Expected runtime: ~15-20 minutes** (35 event/activity models, each doing a full backward-elimination search over ~37 candidate variables). If your kernel session has a shorter time budget, Section 5 can be run one event at a time — see the comment there.

See the companion `Codebook.xlsx` -> **Data Quality Notes** sheet for a full write-up of the two issues fixed below, and the **README** sheet for the R-vs-Python ordered-probit threshold note.

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from statsmodels.miscmodels.ordinal_model import OrderedModel
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import json, os, time

## 1. Load data & fix known data-quality issues

**`hispanic`** in the raw CSV is *not* a clean 0/1 dummy — it's the original survey coding
(1 = Hispanic/Yes, 2 = Not Hispanic/No, plus one ambiguous 0). Using it directly as a regressor
(as the original notebook's `candidate_vars` lists did) silently reverses the sign of the effect,
because higher raw values (2) mean *not* Hispanic. We build a clean `hispanic_c` (1/0/NaN) below.

**`ac`** (has air conditioning) uses **-9** as a skip-logic "not shown" code, not a real 0. The
A/C question is mostly only asked within the Heat module — for Cold/Flooding/Earthquake/Power
Outage respondents, 55-70% have `ac == -9`. Because `-9` is a real integer (not NaN), it survives
`.dropna()` and gets fit as if it were a genuine numeric value whenever `ac` is in a model's
candidate list for those other events. We build a clean `ac_c` (NaN wherever raw is -9) below.

In [ ]:
df = pd.read_csv('./leaphi_BE_distribution_based_dummy.csv', low_memory=False)

# --- hispanic: 1=Hispanic(Yes), 2=Not Hispanic(No), 0=ambiguous(1 case) -> clean 0/1 dummy
df['hispanic_raw'] = df['hispanic']
df['hispanic_c'] = np.where(df['hispanic_raw']==1, 1,
                     np.where(df['hispanic_raw']==2, 0, np.nan))

# --- ac: -9 = skip-logic "Not shown", not a real 0 -> set to missing
df['ac_raw'] = df['ac']
df['ac_c'] = np.where(df['ac_raw']==-9, np.nan, df['ac_raw'])

print('hispanic_c value counts:', df['hispanic_c'].value_counts(dropna=False).to_dict())
print('ac_c missing count:', df['ac_c'].isna().sum(), 'of', len(df))

## 2. Derived variables

### 2a. Severity-level dummies (one set per event)
Each event's `ext_{event}_impact_wlb` (1=Not severe at all ... 5=Extremely severe) is expanded into
four dummies `{event}_imp_2` .. `{event}_imp_5` (level 1 is the reference/omitted category). These
are the "fixed" variables retained in every model, baseline and final.

In [ ]:
EVENT_META = {
    'heat':       dict(flag='ext_heat',       sev='ext_heat_impact_wlb',       label='Extreme Heat'),
    'cold':       dict(flag='ext_cold',       sev='ext_cold_impact_wlb',       label='Extreme Cold'),
    'powerout':   dict(flag='ext_powerout',   sev='ext_powerout_impact_wlb',   label='Power Outage'),
    'earthquake': dict(flag='ext_earthquake', sev='ext_earthquake_impact_wlb', label='Earthquakes'),
    'flooding':   dict(flag='ext_flooding',   sev='ext_flooding_impact_wlb',   label='Flooding'),
}

for ev, meta in EVENT_META.items():
    sev_col = meta['sev']
    for lvl in range(1, 6):
        df[f'{ev}_imp_{lvl}'] = (df[sev_col] == lvl).astype(int)
        df.loc[df[sev_col].isna(), f'{ev}_imp_{lvl}'] = np.nan

print('Severity dummies created for:', list(EVENT_META.keys()))

### 2b. Combined categorical variables

Several independent variables exist in the source data only as sets of 0/1 dummies (e.g. income
is `in50` / `in50100` with `100k+` left as the implicit reference). Per request, each such group is
collapsed into a single ordered categorical column (`*_cat`) — useful for descriptive tables and
for anyone who wants a single ordinal predictor instead of a dummy set.

In [ ]:
# income_cat: 1=<50k, 2=50-100k, 3=100k+, NaN=income not reported
df['income_cat'] = np.select(
    [df['in50']==1, df['in50100']==1, df['in100p']==1], [1,2,3], default=np.nan)

# age_cat: priority age_65p > age_5165 > age_3150 > else 18-30
#   (resolves the ~80 respondents whose raw dummies overlap at exactly age 65)
df['age_cat'] = np.select(
    [df['age_65p']==1, df['age_5165']==1, df['age_3150']==1], [4,3,2], default=1)

# race_cat: Hispanic (any race) > NH Black > NH Asian > NH White > Other NH
#   race and hispanic dummies are NOT mutually exclusive in the raw data, so this uses a priority
#   order (built on the cleaned hispanic_c, not the raw miscoded hispanic column)
df['race_cat'] = np.select(
    [df['hispanic_c']==1, (df['hispanic_c']==0)&(df['black']==1),
     (df['hispanic_c']==0)&(df['asian']==1), (df['hispanic_c']==0)&(df['white']==1)],
    [1,2,3,4], default=5)
df.loc[df['hispanic_c'].isna(), 'race_cat'] = np.nan

# hhsize_cat: 1, 2, 3+
df['hhsize_cat'] = np.select(
    [df['hhsize1']==1, df['hhsize2']==1, df['hhsize3p']==1], [1,2,3], default=np.nan)

# Built-environment 3-bucket variables (low/medium/high)
for be in ['PopDens','EmpDens','NetworkDensity','Diversity','TransitAccess']:
    df[f'{be}_cat'] = np.select(
        [df[f'{be}_low']==1, df[f'{be}_medium']==1, df[f'{be}_high']==1], [1,2,3], default=np.nan)

# NatWalkInd: 4-bucket (very_low/low/high/very_high -- there is no 'medium' bucket for this one)
df['NatWalkInd_cat'] = np.select(
    [df['NatWalkInd_very_low']==1, df['NatWalkInd_low']==1,
     df['NatWalkInd_high']==1, df['NatWalkInd_very_high']==1], [1,2,3,4], default=np.nan)

print('Combined categorical variables created: income_cat, age_cat, race_cat, hhsize_cat,',
      'PopDens_cat, EmpDens_cat, NetworkDensity_cat, Diversity_cat, TransitAccess_cat, NatWalkInd_cat')

## 3. Export five event-level data files

Each file is filtered to respondents **exposed to that event** (`ext_{event}==1`) and contains:
that event's own severity + activity-outcome columns, plus the shared demographic / built-environment
predictors (raw dummies, cleaned variables, and the combined categorical versions). `empsta` is kept
in every file so WFH/WFO models can subset to the working sub-sample downstream — see Section 5.

In [ ]:
os.makedirs('./event_data', exist_ok=True)

In [ ]:
import pandas as pd, numpy as np

EVENT_META = {
    'heat':       dict(flag='ext_heat', sev='ext_heat_impact_wlb'),
    'cold':       dict(flag='ext_cold', sev='ext_cold_impact_wlb'),
    'powerout':   dict(flag='ext_powerout', sev='ext_powerout_impact_wlb'),
    'earthquake': dict(flag='ext_earthquake', sev='ext_earthquake_impact_wlb'),
    'flooding':   dict(flag='ext_flooding', sev='ext_flooding_impact_wlb'),
}

# All raw activity-outcome columns that belong to each event (kept in full, not just the modeled ones)
EVENT_ACTIVITY_COLS = {
    'heat': ['ext_heat_cope','ext_heat_likely_repeat','ext_heat_affect',
             'ext_heat_lkly_normal_business','ext_heat_lkly_stay_home','ext_heat_lkly_seek_shelter',
             'ext_heat_lkly_social_connection','ext_heat_lkly_leave_town','ext_heat_lkly_supplies',
             'ext_heat_lkly_follow_officials','ext_heat_lkly_comm_supports',
             'ext_heat_indoor_restaurant','ext_heat_takeout_pickup','ext_heat_food_delivery',
             'ext_heat_public_indoors','ext_heat_wfh','ext_heat_commute','ext_heat_car_travel',
             'ext_heat_public_transit','ext_heat_stay_home','ext_heat_stay_family_friends',
             'ext_heat_check_family_friends','ext_heat_volunteer_community','ext_heat_ac_equipped'],
    'cold': ['ext_cold_cope','ext_cold_likely_repeat','ext_cold_affect',
             'ext_cold_lkly_normal_business','ext_cold_lkly_stay_home','ext_cold_lkly_seek_shelter',
             'ext_cold_lkly_social_connection','ext_cold_lkly_leave_town','ext_cold_lkly_supplies',
             'ext_cold_lkly_follow_officials','ext_cold_lkly_comm_supports',
             'ext_cold_indoor_restaurant','ext_cold_takeout_pickup','ext_cold_food_delivery',
             'ext_cold_public_indoors','ext_cold_wfh','ext_cold_commute','ext_cold_car_travel',
             'ext_cold_transit_use','ext_cold_stay_home','ext_cold_stay_family_friends',
             'ext_cold_check_family_friends','ext_cold_volunteer_community'],
    'flooding': ['ext_flooding_cope','ext_flooding_likely_repeat','ext_flooding_affect',
             'ext_flooding_lkly_normal_business','ext_flooding_lkly_stay_home','ext_flooding_lkly_seek_shelter',
             'ext_flooding_lkly_social_connection','ext_flooding_lkly_leave_town','ext_flooding_lkly_supplies',
             'ext_flooding_lkly_follow_officials','ext_flooding_lkly_comm_supports',
             'ext_flooding_wfh','ext_flooding_car_travel','ext_flooding_walk_bike_mode',
             'ext_flooding_transit_use','ext_flooding_stay_home','ext_flooding_stay_family_friends',
             'ext_flooding_check_family_friends','ext_flooding_volunteer_community'],
    'earthquake': ['ext_earthquake_cope','ext_earthquake_likely_repeat','ext_earthquake_affect',
             'ext_earthquake_lkly_normal_business','ext_earthquake_lkly_stay_home','ext_earthquake_lkly_seek_shelter',
             'ext_earthquake_lkly_social_connection','ext_earthquake_lkly_leave_town','ext_earthquake_lkly_supplies',
             'ext_earthquake_lkly_follow_officials','ext_earthquake_lkly_comm_supports',
             'ext_earthquake_wfh','ext_earthquake_car_travel','ext_earthquake_walk_bike_mode',
             'ext_earthquake_transit_use','ext_earthquake_stay_home','ext_earthquake_stay_family_friends',
             'ext_earthquake_check_family_friends','ext_earthquake_volunteer_community'],
    'powerout': ['ext_powerout_cope','ext_powerout_likely_repeat','ext_powerout_affect',
             'ext_powerout_lkly_normal_business','ext_powerout_lkly_stay_home','ext_powerout_lkly_seek_shelter',
             'ext_powerout_lkly_social_connection','ext_powerout_lkly_leave_town','ext_powerout_lkly_supplies',
             'ext_powerout_lkly_follow_officials','ext_powerout_lkly_comm_supports',
             'ext_powerout_eating_restaurant','ext_powerout_indoor_restaurant','ext_powerout_food_delivery',
             'ext_powerout_takeout_pickup','ext_powerout_car_travel','ext_powerout_walk_bike_mode',
             'ext_powerout_transit_use','ext_powerout_stay_home','ext_powerout_stay_family_friends',
             'ext_powerout_check_family_friends','ext_powerout_volunteer_community'],
}

IV_COLS = ["CR","SE","PR","female","age_3150","age_5165","age_65p","age_cat",
           "edu_bs","bs_grad","white","black","asian","hispanic_raw","hispanic_c","race_cat",
           "in50","in50100","income_cat","hhsize1","hhsize2","hhsize_cat","child","dis_yes",
           "work_out","tcom_no","sa_home","hhveh0","ac_raw","ac_c","rural",
           "PopDens_low","PopDens_medium","PopDens_high","PopDens_cat",
           "EmpDens_low","EmpDens_medium","EmpDens_high","EmpDens_cat",
           "NetworkDensity_low","NetworkDensity_medium","NetworkDensity_high","NetworkDensity_cat",
           "Diversity_low","Diversity_medium","Diversity_high","Diversity_cat",
           "TransitAccess_low","TransitAccess_medium","TransitAccess_high","TransitAccess_cat",
           "NatWalkInd_very_low","NatWalkInd_low","NatWalkInd_high","NatWalkInd_very_high","NatWalkInd_cat"]

ID_COLS = ["ResponseId","state","empsta","weights","hcity","zip_code","urban","suburban","rural"]

for ev, meta in EVENT_META.items():
    sub = df[df[meta['flag']]==1].copy()
    sev_dummies = [f'{ev}_imp_{i}' for i in range(1,6)]
    cols = ID_COLS + [meta['sev']] + sev_dummies + EVENT_ACTIVITY_COLS[ev] + IV_COLS
    cols = [c for c in dict.fromkeys(cols) if c in sub.columns]  # dedupe, keep existing only
    out = sub[cols].copy()
    path = f'./event_data/{ev}_data.csv'
    out.to_csv(path, index=False)
    print(ev, out.shape, '->', path)


## 4. Ordered probit modeling functions

Generalized, reusable versions of the fitting / backward-elimination code that was repeated (with small inconsistencies) across many cells of the original notebook.

In [ ]:
import pandas as pd, numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
import warnings
warnings.filterwarnings('ignore')

CANDIDATE_VARS = ["CR","SE","PR","female","age_3150","age_5165","age_65p",
                   "edu_bs","bs_grad","white","black","asian","hispanic_c",
                   "in50","in50100","hhsize1","hhsize2","child","dis_yes",
                   "work_out","tcom_no","sa_home","hhveh0","ac_c","rural",
                   "PopDens_medium","PopDens_high","EmpDens_medium","EmpDens_high",
                   "NetworkDensity_medium","NetworkDensity_high",
                   "Diversity_medium","Diversity_high",
                   "TransitAccess_medium","TransitAccess_high",
                   "NatWalkInd_low","NatWalkInd_high","NatWalkInd_very_high"]

def fit_ordered_probit(data, y_var, x_vars):
    """Fit an ordered probit and return a tidy coefficient table with PROPERLY
    transformed threshold/cutpoints (statsmodels stores raw thresholds internally;
    all but the first must be passed through transform_threshold_params to be
    comparable to R's polr/clm zetas)."""
    model_data = data[x_vars + [y_var]].dropna()
    if model_data[y_var].nunique() < 2 or len(model_data) < (len(x_vars) + 10):
        return None, model_data.shape[0]
    y = model_data[y_var]
    X = model_data[x_vars]
    try:
        model = OrderedModel(y, X, distr='probit')
        res = model.fit(method='bfgs', disp=False, maxiter=200)
    except Exception:
        return None, model_data.shape[0]

    n_thresh = model.k_levels - 1
    beta = res.params[:-n_thresh]
    beta_se = res.bse[:-n_thresh]
    beta_z = res.tvalues[:-n_thresh]
    beta_p = res.pvalues[:-n_thresh]

    thresh_raw = res.params[-n_thresh:]
    thresh = model.transform_threshold_params(res.params.values)[1:-1]  # drop -inf/+inf

    rows = []
    for v in x_vars:
        p = beta_p[v]
        rows.append(dict(Variable=v, Coef=beta[v], SE=beta_se[v], z=beta_z[v], p=p,
                          sig='***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '+' if p<0.1 else ''))
    for i, t in enumerate(thresh):
        rows.append(dict(Variable=f'threshold_{i+1}', Coef=t, SE=np.nan, z=np.nan, p=np.nan, sig=''))
    out = pd.DataFrame(rows)
    out.attrs['nobs'] = int(res.nobs)
    out.attrs['llf'] = res.llf
    out.attrs['aic'] = res.aic
    out.attrs['prsquared'] = res.prsquared
    return out, model_data.shape[0]

def stepwise_backward(data, y_var, fixed_vars, candidate_vars, sig_level=0.1):
    """Backward elimination on candidate_vars only; fixed_vars always retained."""
    current = list(candidate_vars)
    model_data_full = data[fixed_vars + current + [y_var]].dropna()
    if model_data_full[y_var].nunique() < 2:
        return None, None, model_data_full.shape[0]
    y = model_data_full[y_var]
    guard = 0
    while current and guard < 60:
        guard += 1
        X = model_data_full[fixed_vars + current]
        try:
            res = OrderedModel(y, X, distr='probit').fit(method='bfgs', disp=False, maxiter=200)
        except Exception:
            return None, None, model_data_full.shape[0]
        pvals = res.pvalues[current]
        worst = pvals.idxmax()
        if pvals[worst] < sig_level:
            break
        current.remove(worst)
        if not current:
            break
    final_vars = fixed_vars + current
    final_tbl, n = fit_ordered_probit(data, y_var, final_vars)
    return final_tbl, current, n




### A note on thresholds (why R and Python cutpoints differ)

`statsmodels.OrderedModel` does **not** store raw threshold/cutpoint values directly in
`results.params` — everything except the first threshold is stored as a *log-difference*
internally, purely so BFGS can't produce a non-increasing set of cutpoints during optimization.
Comparing those raw numbers to R's `polr()`/`clm()` zetas will never match. `fit_ordered_probit()`
above calls `model.transform_threshold_params()` to recover the real, R-comparable cutpoints —
that's what gets reported as `threshold_1`, `threshold_2`, `threshold_3`, `threshold_4` in the
output tables. Your regression coefficients (the X variables) do not need this correction; both
packages already use the same `P(Y<=j) = F(threshold_j - X*beta)` convention.

## 5. Run all models

Baseline (severity only) + backward-elimination final model (p < 0.10) for every Event x Activity
combination that exists in the data (35 total: 9 for Heat, 9 for Cold, 7 for Power Outage, 5 for
Earthquake, 5 for Flooding — Power Outage has no WFH/WFO question; Earthquake/Flooding have neither
WFO nor the food-related activities).

WFH and WFO models restrict to the currently-working sub-sample (`empsta < 3`); every other
activity uses the full event-exposed sample.

> **If ~15-20 minutes is too long for one run:** loop over `ACTIVITIES.items()` for a single event
> at a time (e.g. set `EVENTS_TO_RUN = ['powerout']`), and merge the resulting `results` dicts.

In [ ]:
CANDIDATE_VARS_STD = CANDIDATE_VARS  # standard 37-variable candidate list, defined above

ACTIVITIES = {
    'Usual':    {'heat':('ext_heat_lkly_normal_business',False), 'cold':('ext_cold_lkly_normal_business',False),
                 'powerout':('ext_powerout_lkly_normal_business',False), 'earthquake':('ext_earthquake_lkly_normal_business',False),
                 'flooding':('ext_flooding_lkly_normal_business',False)},
    'Home':     {'heat':('ext_heat_stay_home',False), 'cold':('ext_cold_stay_home',False),
                 'powerout':('ext_powerout_stay_home',False), 'earthquake':('ext_earthquake_stay_home',False),
                 'flooding':('ext_flooding_stay_home',False)},
    'Car':      {'heat':('ext_heat_car_travel',False), 'cold':('ext_cold_car_travel',False),
                 'powerout':('ext_powerout_car_travel',False), 'earthquake':('ext_earthquake_car_travel',False),
                 'flooding':('ext_flooding_car_travel',False)},
    'Transit':  {'heat':('ext_heat_public_transit',False), 'cold':('ext_cold_transit_use',False),
                 'powerout':('ext_powerout_transit_use',False), 'earthquake':('ext_earthquake_transit_use',False),
                 'flooding':('ext_flooding_transit_use',False)},
    'WFH':      {'heat':('ext_heat_wfh',True), 'cold':('ext_cold_wfh',True),
                 'flooding':('ext_flooding_wfh',True), 'earthquake':('ext_earthquake_wfh',True)},
    'WFO':      {'heat':('ext_heat_commute',True), 'cold':('ext_cold_commute',True)},
    'Dine in':  {'heat':('ext_heat_indoor_restaurant',False), 'cold':('ext_cold_indoor_restaurant',False),
                 'powerout':('ext_powerout_indoor_restaurant',False)},
    'Pick up':  {'heat':('ext_heat_takeout_pickup',False), 'cold':('ext_cold_takeout_pickup',False),
                 'powerout':('ext_powerout_takeout_pickup',False)},
    'Delivery': {'heat':('ext_heat_food_delivery',False), 'cold':('ext_cold_food_delivery',False),
                 'powerout':('ext_powerout_food_delivery',False)},
}

results = {}
for act, evmap in ACTIVITIES.items():
    for ev, (dv, worker_only) in evmap.items():
        t0 = time.time()
        sub = df[df[EVENT_META[ev]['flag']]==1].copy()
        if worker_only:
            sub = sub[sub['empsta']<3]
        fixed = [f'{ev}_imp_{lvl}' for lvl in [2,3,4,5]]
        base_tbl, n_base = fit_ordered_probit(sub, dv, fixed)
        final_tbl, kept, n_final = stepwise_backward(sub, dv, fixed, CANDIDATE_VARS_STD, sig_level=0.1)
        results[(act, ev)] = dict(baseline=base_tbl, final=final_tbl, kept=kept, n_base=n_base, n_final=n_final)
        status = 'OK' if final_tbl is not None else 'FAILED'
        print(f'{act:10s} {ev:10s} n_base={n_base:5d} n_final={n_final:5d} '
              f'kept={len(kept) if kept is not None else 0:2d} [{status}] {time.time()-t0:.1f}s')

print(f'\nCompleted {len(results)} models.')

## 6. Export consolidated Model_Results.xlsx

One sheet per activity; within each sheet, BASELINE and FINAL MODEL blocks for every event that has that activity.

In [ ]:
EVENT_ORDER = ['heat','cold','powerout','earthquake','flooding']
EVENT_LABEL = {'heat':'Extreme Heat','cold':'Extreme Cold','powerout':'Power Outage',
               'earthquake':'Earthquakes','flooding':'Flooding'}
ACT_ORDER = ['Usual','Home','Car','Transit','WFH','WFO','Dine in','Pick up','Delivery']

SEV_LABELS = {1:'Slightly severe',2:'Moderately severe',3:'Very severe',4:'Extremely severe'}
# fixed vars are named {event}_imp_2..5 -> map position 2,3,4,5 to labels above (imp_2->Slightly, imp_3->Moderately, imp_4->Very, imp_5->Extremely)
IMP_POS_LABEL = {2:'Slightly severe',3:'Moderately severe',4:'Very severe',5:'Extremely severe'}

FONT = 'Arial'
bold = Font(name=FONT, bold=True)
normal = Font(name=FONT)
header_fill = PatternFill('solid', fgColor='1F4E78')
header_font = Font(name=FONT, bold=True, color='FFFFFF')
section_fill = PatternFill('solid', fgColor='D9E1F2')
thin = Side(style='thin', color='BFBFBF')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_block(ws, row, tbl, fixed_prefix, label):
    """Write one model's coefficient table starting at `row`. Returns next free row."""
    ws.cell(row=row, column=1, value=label).font = bold
    ws.cell(row=row, column=1).fill = section_fill
    for c in range(2,7):
        ws.cell(row=row, column=c).fill = section_fill
    row += 1
    headers = ['Variable','Coef','S.E.','z','p-value','Sig.']
    for c, h in enumerate(headers, start=1):
        cell = ws.cell(row=row, column=c, value=h)
        cell.font = header_font
        cell.fill = header_fill
        cell.border = border
    row += 1
    if tbl is None:
        ws.cell(row=row, column=1, value='(model did not converge / insufficient data)').font = normal
        return row + 2
    for _, r in tbl.iterrows():
        var = r['Variable']
        if var.startswith(fixed_prefix + '_imp_'):
            lvl = int(var.split('_')[-1])
            disp = IMP_POS_LABEL.get(lvl, var)
        elif var.startswith('threshold_'):
            disp = 'Threshold ' + var.split('_')[-1] + ' (cutpoint, transformed)'
        elif var == 'hispanic_c':
            disp = 'hispanic (cleaned 0/1)'
        elif var == 'ac_c':
            disp = 'ac (cleaned, -9 set to missing)'
        else:
            disp = var
        vals = [disp, r['Coef'], r['SE'], r['z'], r['p'], r['sig']]
        for c, v in enumerate(vals, start=1):
            cell = ws.cell(row=row, column=c, value=(round(v,4) if isinstance(v,(int,float)) and not pd.isna(v) else (None if isinstance(v,float) and pd.isna(v) else v)))
            cell.font = normal
            cell.border = border
        row += 1
    n = tbl.attrs.get('nobs','')
    llf = tbl.attrs.get('llf', np.nan)
    aic = tbl.attrs.get('aic', np.nan)
    pr2 = tbl.attrs.get('prsquared', np.nan)
    ws.cell(row=row, column=1, value=f"N = {n}   Log-L = {llf:.2f}   AIC = {aic:.2f}   Pseudo R² = {pr2:.4f}").font = Font(name=FONT, italic=True, size=9)
    row += 2
    return row

wb = Workbook()
wb.remove(wb.active)

for act in ACT_ORDER:
    combos = [(act, ev) for ev in EVENT_ORDER if (act, ev) in results]
    if not combos:
        continue
    ws = wb.create_sheet(title=act[:31])
    ws.column_dimensions['A'].width = 34
    for c in 'BCDEF':
        ws.column_dimensions[c].width = 12
    row = 1
    ws.cell(row=row, column=1, value=f"Activity: {act}").font = Font(name=FONT, bold=True, size=13)
    row += 2
    for ev in EVENT_ORDER:
        if (act, ev) not in results:
            continue
        res = results[(act, ev)]
        row = write_block(ws, row, res['baseline'], ev, f"{EVENT_LABEL[ev]} — BASELINE (severity only)")
        row = write_block(ws, row, res['final'], ev, f"{EVENT_LABEL[ev]} — FINAL MODEL (severity + significant controls, backward elimination, p<0.10)")
        row += 1

wb.save('./Model_Results.xlsx')
print("saved Model_Results.xlsx")


## 7. Codebook

Builds `Codebook.xlsx` documenting every dependent variable (by event), every independent variable
(raw + cleaned), the combined categorical variables, event sample definitions, and the two data
quality issues fixed in Section 1. See the standalone `Codebook.xlsx` delivered alongside this
notebook for the full, already-generated version; the cell below regenerates it from scratch so the
whole pipeline — data, models, and documentation — lives in one runnable notebook.

In [ ]:
orig = {}  # original source codebook labels are folded directly into the tables below

FONT='Arial'
header_font = Font(name=FONT, bold=True, color='FFFFFF')
header_fill = PatternFill('solid', fgColor='1F4E78')
title_font = Font(name=FONT, bold=True, size=14)
sub_font = Font(name=FONT, italic=True, size=10, color='555555')
normal = Font(name=FONT, size=10)
bold = Font(name=FONT, bold=True, size=10)
thin = Side(style='thin', color='D9D9D9')
border = Border(left=thin, right=thin, top=thin, bottom=thin)
wrap = Alignment(wrap_text=True, vertical='top')

def style_header(ws, row, headers, widths):
    for c, (h,w) in enumerate(zip(headers,widths), start=1):
        cell = ws.cell(row=row, column=c, value=h)
        cell.font = header_font; cell.fill = header_fill; cell.alignment = Alignment(vertical='center')
        ws.column_dimensions[get_column_letter(c)].width = w
    ws.freeze_panes = ws.cell(row=row+1, column=1).coordinate

def codes_str(var):
    if var in orig and orig[var]['codes']:
        return '; '.join(f"{c}={l}" for c,l in orig[var]['codes'])
    return ''

wb = Workbook()

# ---------------- README ----------------
ws = wb.active
ws.title = 'README'
ws.column_dimensions['A'].width = 100
r=1
ws.cell(row=r, column=1, value='LEAP-HI Extreme Weather Travel Behavior — Data Codebook').font = title_font
r+=2
lines = [
 "This workbook documents every variable used in the ordered probit modeling of how five extreme-weather/",
 "disruption events (Extreme Heat, Extreme Cold, Power Outage, Earthquake, Flooding) affect travel and daily",
 "activity choices. Source data: leaphi_BE_distribution_based_dummy.csv (N=5,089 respondents, 451 raw columns).",
 "",
 "SHEETS IN THIS WORKBOOK",
 "  1. README                     — this page",
 "  2. Dependent Variables         — severity/impact ratings + activity-change outcomes, by event",
 "  3. Independent Variables       — demographic, household, and built-environment predictors used in models",
 "  4. Combined Categorical Vars   — derived multi-level variables built by collapsing groups of 0/1 dummies",
 "  5. Event Sample Definitions    — which respondents/rows go into each event's data file, and why",
 "  6. Data Quality Notes          — two coding issues found in the raw CSV and how they were corrected",
 "",
 "MODEL SPECIFICATION",
 "  Estimator: ordered probit (statsmodels OrderedModel, distr='probit'), no intercept (thresholds absorb it).",
 "  For each Event x Activity outcome, two models are reported in Model_Results.xlsx:",
 "    BASELINE    = activity ~ severity dummies only (4 dummies, 'Not severe at all' is the reference level)",
 "    FINAL MODEL = severity dummies (always retained) + candidate controls surviving backward elimination",
 "                  (p < 0.10), starting from the standard candidate list in the 'Independent Variables' sheet.",
 "  Samples: WFH and WFO (commute) models restrict to respondents with empsta < 3 (currently working, in",
 "  some capacity); all other activities use everyone exposed to that event. See 'Event Sample Definitions'.",
 "",
 "A NOTE ON THRESHOLDS / CUTPOINTS",
 "  statsmodels does not store raw threshold parameters directly comparable to R's polr()/clm() zetas — all",
 "  but the first threshold are stored as log-differences internally to keep them increasing during",
 "  optimization. Model_Results.xlsx reports the properly transformed thresholds (via",
 "  model.transform_threshold_params()), which ARE comparable to R output. If you see a mismatch against an",
 "  earlier R run, this reparameterization is the most likely reason — check whether the R-side thresholds",
 "  were read off raw params or off predict()/summary() output.",
]
for line in lines:
    ws.cell(row=r, column=1, value=line).font = bold if line.isupper() or (line and line==line.upper() and not line.startswith(' ')) else normal
    r+=1

# ---------------- Dependent Variables ----------------
ws = wb.create_sheet('Dependent Variables')
style_header(ws, 1, ['Event','Variable','Description','Value Coding','Role','Notes'],
             [14,34,34,55,16,40])
r=2
EVENTS = [('Extreme Heat','heat','ext_heat'), ('Extreme Cold','cold','ext_cold'),
          ('Power Outage','powerout','ext_powerout'), ('Earthquake','earthquake','ext_earthquake'),
          ('Flooding','flooding','ext_flooding')]

ACTIVITY_VARS = {
 'heat': [('ext_heat_impact_wlb','Severity of impact on daily life last time this event happened','severity (fixed var in all models)'),
          ('ext_heat_lkly_normal_business','Likelihood of going about business as usual','Usual'),
          ('ext_heat_stay_home','Change in staying at home','Home'),
          ('ext_heat_car_travel','Change in using a car for traveling','Car'),
          ('ext_heat_public_transit','Change in taking public transit','Transit'),
          ('ext_heat_wfh','Change in working from home','WFH'),
          ('ext_heat_commute','Change in working from the office','WFO'),
          ('ext_heat_indoor_restaurant','Change in eating indoors at a restaurant','Dine in'),
          ('ext_heat_takeout_pickup','Change in picking up takeout','Pick up'),
          ('ext_heat_food_delivery','Change in having food delivered','Delivery'),
          ('ext_heat_cope','Coping strategy last time this event happened','not modeled'),
          ('ext_heat_likely_repeat','Likelihood of experiencing this event again','not modeled'),
          ('ext_heat_affect','Overall degree to which routine was affected','not modeled'),
          ('ext_heat_public_indoors','Change in spending time in public indoor spaces','not modeled'),
          ('ext_heat_stay_family_friends','Change in staying with family/friends','not modeled'),
          ('ext_heat_check_family_friends','Change in checking on family/friends','not modeled'),
          ('ext_heat_volunteer_community','Change in volunteering in the community','not modeled'),
          ('ext_heat_ac_equipped','Whether home is equipped with air conditioning','not modeled')],
 'cold': [('ext_cold_impact_wlb','Severity of impact on daily life last time this event happened','severity (fixed var in all models)'),
          ('ext_cold_lkly_normal_business','Likelihood of going about business as usual','Usual'),
          ('ext_cold_stay_home','Change in staying at home','Home'),
          ('ext_cold_car_travel','Change in using a car for traveling','Car'),
          ('ext_cold_transit_use','Change in taking public transit','Transit'),
          ('ext_cold_wfh','Change in working from home','WFH'),
          ('ext_cold_commute','Change in working from the office','WFO'),
          ('ext_cold_indoor_restaurant','Change in eating indoors at a restaurant','Dine in'),
          ('ext_cold_takeout_pickup','Change in picking up takeout','Pick up'),
          ('ext_cold_food_delivery','Change in having food delivered','Delivery'),
          ('ext_cold_cope','Coping strategy last time this event happened','not modeled'),
          ('ext_cold_likely_repeat','Likelihood of experiencing this event again','not modeled'),
          ('ext_cold_affect','Overall degree to which routine was affected','not modeled'),
          ('ext_cold_public_indoors','Change in spending time in public indoor spaces','not modeled'),
          ('ext_cold_stay_family_friends','Change in staying with family/friends','not modeled'),
          ('ext_cold_check_family_friends','Change in checking on family/friends','not modeled'),
          ('ext_cold_volunteer_community','Change in volunteering in the community','not modeled')],
 'powerout': [('ext_powerout_impact_wlb','Severity of impact on daily life last time this event happened','severity (fixed var in all models)'),
          ('ext_powerout_lkly_normal_business','Likelihood of going about business as usual','Usual'),
          ('ext_powerout_stay_home','Change in staying at home','Home'),
          ('ext_powerout_car_travel','Change in using a car for traveling','Car'),
          ('ext_powerout_transit_use','Change in taking public transit','Transit'),
          ('ext_powerout_indoor_restaurant','Change in eating indoors at a restaurant','Dine in'),
          ('ext_powerout_takeout_pickup','Change in picking up takeout','Pick up'),
          ('ext_powerout_food_delivery','Change in having food delivered','Delivery'),
          ('ext_powerout_cope','Coping strategy last time this event happened','not modeled'),
          ('ext_powerout_likely_repeat','Likelihood of experiencing this event again','not modeled'),
          ('ext_powerout_affect','Overall degree to which routine was affected','not modeled'),
          ('ext_powerout_eating_restaurant','Change in eating at a restaurant (general)','not modeled'),
          ('ext_powerout_walk_bike_mode','Change in walking/biking','not modeled'),
          ('ext_powerout_stay_family_friends','Change in staying with family/friends','not modeled'),
          ('ext_powerout_check_family_friends','Change in checking on family/friends','not modeled'),
          ('ext_powerout_volunteer_community','Change in volunteering in the community','not modeled'),
          ],
 'earthquake': [('ext_earthquake_impact_wlb','Severity of impact on daily life last time this event happened','severity (fixed var in all models)'),
          ('ext_earthquake_lkly_normal_business','Likelihood of going about business as usual','Usual'),
          ('ext_earthquake_stay_home','Change in staying at home','Home'),
          ('ext_earthquake_car_travel','Change in using a car for traveling','Car'),
          ('ext_earthquake_transit_use','Change in taking public transit','Transit'),
          ('ext_earthquake_wfh','Change in working from home','WFH'),
          ('ext_earthquake_cope','Coping strategy last time this event happened','not modeled'),
          ('ext_earthquake_likely_repeat','Likelihood of experiencing this event again','not modeled'),
          ('ext_earthquake_affect','Overall degree to which routine was affected','not modeled'),
          ('ext_earthquake_walk_bike_mode','Change in walking/biking','not modeled'),
          ('ext_earthquake_stay_family_friends','Change in staying with family/friends','not modeled'),
          ('ext_earthquake_check_family_friends','Change in checking on family/friends','not modeled'),
          ('ext_earthquake_volunteer_community','Change in volunteering in the community','not modeled')],
 'flooding': [('ext_flooding_impact_wlb','Severity of impact on daily life last time this event happened','severity (fixed var in all models)'),
          ('ext_flooding_lkly_normal_business','Likelihood of going about business as usual','Usual'),
          ('ext_flooding_stay_home','Change in staying at home','Home'),
          ('ext_flooding_car_travel','Change in using a car for traveling','Car'),
          ('ext_flooding_transit_use','Change in taking public transit','Transit'),
          ('ext_flooding_wfh','Change in working from home','WFH'),
          ('ext_flooding_cope','Coping strategy last time this event happened','not modeled'),
          ('ext_flooding_likely_repeat','Likelihood of experiencing this event again','not modeled'),
          ('ext_flooding_affect','Overall degree to which routine was affected','not modeled'),
          ('ext_flooding_walk_bike_mode','Change in walking/biking','not modeled'),
          ('ext_flooding_stay_family_friends','Change in staying with family/friends','not modeled'),
          ('ext_flooding_check_family_friends','Change in checking on family/friends','not modeled'),
          ('ext_flooding_volunteer_community','Change in volunteering in the community','not modeled')],
}

for label, ev, flag in EVENTS:
    for var, desc, role in ACTIVITY_VARS[ev]:
        codes = codes_str(var)
        documented = var in orig
        if not codes:
            codes = '-9=Not shown (skip logic); ' + ('documented range: ' + str(sorted(df[var].dropna().unique().tolist())) if var in df.columns else 'n/a')
        notes = '' if documented else 'Value labels not in source codebook — verify against survey instrument before using outside this project.'
        row = [label, var, desc, codes, role, notes]
        for c, v in enumerate(row, start=1):
            cell = ws.cell(row=r, column=c, value=v); cell.font = normal; cell.border = border; cell.alignment = wrap
        r += 1
    # severity dummy rows
    for lvl, lvl_label in [(2,'Slightly severe'),(3,'Moderately severe'),(4,'Very severe'),(5,'Extremely severe')]:
        var = f'{ev}_imp_{lvl}'
        row = [label, var, f'Derived dummy = 1 if {ev}_impact_wlb == {lvl}, else 0 (NaN if severity missing)',
               f'1 = {lvl_label}; 0 = otherwise', 'severity dummy (fixed var)', 'Derived; reference level is imp_1 ("Not severe at all")']
        for c, v in enumerate(row, start=1):
            cell = ws.cell(row=r, column=c, value=v); cell.font = normal; cell.border = border; cell.alignment = wrap
        r += 1

# ---------------- Independent Variables ----------------
ws = wb.create_sheet('Independent Variables')
style_header(ws, 1, ['Variable','Description','Value Coding / Range','Type','Notes'], [22,36,50,14,45])
r = 2
IV_ROWS = [
 ('CR', 'Community resilience — standardized factor score from resilience-attitude items', 'Continuous, mean≈0, SD≈1 (range ≈ -3.5 to +2.3)', 'continuous', 'From factor analysis (factor_analyzer); higher = greater community resilience.'),
 ('SE', 'Social engagement — standardized factor score', 'Continuous, mean≈0, SD≈1', 'continuous', 'Same construction as CR.'),
 ('PR', 'Personal resilience — standardized factor score', 'Continuous, mean≈0, SD≈1', 'continuous', 'Same construction as CR.'),
 ('female', 'Respondent is female', '1=Female; 0=Not female', 'dummy', ''),
 ('age_3150', 'Respondent age 31–50', '1=Yes; 0=No', 'dummy', 'Reference group across the age dummies = 18–30.'),
 ('age_5165', 'Respondent age 51–65', '1=Yes; 0=No', 'dummy', ''),
 ('age_65p', 'Respondent age 65+', '1=Yes; 0=No', 'dummy', ''),
 ('edu_bs', "Has a Bachelor's degree", '1=Yes; 0=No', 'dummy', ''),
 ('bs_grad', 'Has at least a Bachelor\'s degree (BS or higher)', '1=Yes; 0=No', 'dummy', ''),
 ('white', 'Race = White (may co-occur with Hispanic ethnicity)', '1=Yes; 0=No', 'dummy', 'Not mutually exclusive with hispanic — see race_cat for a single combined category.'),
 ('black', 'Race = Black', '1=Yes; 0=No', 'dummy', ''),
 ('asian', 'Race = Asian', '1=Yes; 0=No', 'dummy', ''),
 ('hispanic_c', 'CLEANED Hispanic ethnicity dummy — use this, not raw hispanic', '1=Hispanic; 0=Not Hispanic; NaN=ambiguous (1 respondent)', 'dummy (cleaned)', 'See "Data Quality Notes" sheet — raw hispanic column is 1=Yes/2=No, not 0/1.'),
 ('hispanic_raw', 'Raw survey column, kept for traceability only — do not use directly in models', '1=Hispanic (Yes); 2=Not Hispanic (No); 0=ambiguous (1 case)', 'raw / do not use', 'Using this directly as a numeric regressor reverses the sign of the effect.'),
 ('in50', 'Household income under $50k', '1=Yes; 0=No', 'dummy', 'Reference group = $100k+.'),
 ('in50100', 'Household income $50k–$100k', '1=Yes; 0=No', 'dummy', ''),
 ('hhsize1', 'One-person household', '1=Yes; 0=No', 'dummy', 'Reference group = 3+ person household.'),
 ('hhsize2', 'Two-person household', '1=Yes; 0=No', 'dummy', ''),
 ('child', 'Household has a child', '1=Yes; 0=No', 'dummy', ''),
 ('dis_yes', 'Respondent has a travel-related disability', '1=Yes; 0=No', 'dummy', ''),
 ('work_out', 'Works primarily outdoors', '1=Yes; 0=No', 'dummy', ''),
 ('tcom_no', 'Does not telecommute', '1=Yes; 0=No', 'dummy', ''),
 ('sa_home', 'Lives in a stand-alone / detached house', '1=Yes; 0=No', 'dummy', ''),
 ('hhveh0', 'Zero-vehicle household', '1=Yes; 0=No', 'dummy', ''),
 ('ac_c', 'CLEANED has-air-conditioner dummy — use this, not raw ac', '1=Yes; 0=No; NaN=not asked (skip logic)', 'dummy (cleaned)', 'See "Data Quality Notes" — raw ac uses -9 for "not shown", not a real 0. Mostly only asked of heat-exposed respondents.'),
 ('ac_raw', 'Raw survey column, kept for traceability only — do not use directly in models', '1=Yes; 0=No; -9=Not shown (skip logic)', 'raw / do not use', 'Treating -9 as a numeric value contaminates any model outside the Heat subsample.'),
 ('rural', 'Lives in a rural area', '1=Yes; 0=No', 'dummy', ''),
 ('PopDens_medium', 'Population density: medium tercile (>4.00–8.86 people/acre)', '1=Yes; 0=No', 'dummy', 'Reference = low.'),
 ('PopDens_high', 'Population density: high tercile (>8.86 people/acre)', '1=Yes; 0=No', 'dummy', ''),
 ('EmpDens_medium', 'Employment density: medium tercile (>0.91–2.37 jobs/acre)', '1=Yes; 0=No', 'dummy', 'Reference = low.'),
 ('EmpDens_high', 'Employment density: high tercile (>2.37 jobs/acre)', '1=Yes; 0=No', 'dummy', ''),
 ('NetworkDensity_medium', 'Street network density: medium tercile', '1=Yes; 0=No', 'dummy', 'Reference = low. NOTE: in the source file, NetworkDensity and Diversity thresholds are swapped in the field labels — verify against your GIS source if precise cutoffs matter.'),
 ('NetworkDensity_high', 'Street network density: high tercile', '1=Yes; 0=No', 'dummy', ''),
 ('Diversity_medium', 'Land-use diversity: medium tercile', '1=Yes; 0=No', 'dummy', 'Reference = low.'),
 ('Diversity_high', 'Land-use diversity: high tercile', '1=Yes; 0=No', 'dummy', ''),
 ('TransitAccess_medium', 'Transit access: medium tercile (>0–0.13)', '1=Yes; 0=No', 'dummy', 'Reference = low (0).'),
 ('TransitAccess_high', 'Transit access: high tercile (>0.13)', '1=Yes; 0=No', 'dummy', ''),
 ('NatWalkInd_low', 'National Walkability Index: low quartile (>8–10)', '1=Yes; 0=No', 'dummy', 'Reference = very low (≤8).'),
 ('NatWalkInd_high', 'National Walkability Index: high quartile (>10–13)', '1=Yes; 0=No', 'dummy', ''),
 ('NatWalkInd_very_high', 'National Walkability Index: very high quartile (>13)', '1=Yes; 0=No', 'dummy', ''),
 ('empsta', 'Employment status', '1=Both worker & student; 2=Worker; 3=Student; 4=Neither', 'categorical', 'WFH/WFO models restrict to empsta<3 (i.e. currently employed).'),
 ('weights', 'Survey weight', 'continuous', 'weight', 'Present in source data; not currently applied in the estimation code shown.'),
]
for row in IV_ROWS:
    for c, v in enumerate(row, start=1):
        cell = ws.cell(row=r, column=c, value=v); cell.font = normal; cell.border = border; cell.alignment = wrap
    r += 1

# ---------------- Combined Categorical Variables ----------------
ws = wb.create_sheet('Combined Categorical Vars')
style_header(ws, 1, ['Variable','Built from','Value Coding','Notes'], [18,40,55,55])
r=2
CAT_ROWS = [
 ('income_cat', 'in50, in50100, in100p', '1=<$50k; 2=$50k-$100k; 3=$100k+; NaN=income not reported', '156 respondents did not report income (NaN).'),
 ('age_cat', 'age_3150, age_5165, age_65p', '1=18-30; 2=31-50; 3=51-65; 4=65+', "80 respondents' raw dummies overlapped at exactly age 65 (satisfy both age_5165 and age_65p); resolved by giving age_65p priority."),
 ('race_cat', 'hispanic_c, white, black, asian', '1=Hispanic (any race); 2=Non-Hispanic Black; 3=Non-Hispanic Asian; 4=Non-Hispanic White; 5=Other/multiracial, non-Hispanic; NaN=hispanic_c missing',
    'Priority order Hispanic > Black > Asian > White > Other, since race and Hispanic-ethnicity dummies are not mutually exclusive in the raw data (someone can be both White and Hispanic). Uses the CLEANED hispanic_c, not the raw miscoded column.'),
 ('hhsize_cat', 'hhsize1, hhsize2, hhsize3p', '1=1 person; 2=2 people; 3=3+ people', ''),
 ('PopDens_cat', 'PopDens_low/medium/high', '1=Low (≤4.00 ppl/acre); 2=Medium (>4.00-8.86); 3=High (>8.86); NaN=BE data unmatched', '162 respondents could not be matched to built-environment data (be_unmatched=1).'),
 ('EmpDens_cat', 'EmpDens_low/medium/high', '1=Low (≤0.91 jobs/acre); 2=Medium (>0.91-2.37); 3=High (>2.37); NaN=BE data unmatched', 'Same 162 respondents missing.'),
 ('NetworkDensity_cat', 'NetworkDensity_low/medium/high', '1=Low; 2=Medium; 3=High; NaN=BE data unmatched', 'Same 162 respondents missing.'),
 ('Diversity_cat', 'Diversity_low/medium/high', '1=Low; 2=Medium; 3=High; NaN=BE data unmatched', 'Same 162 respondents missing.'),
 ('TransitAccess_cat', 'TransitAccess_low/medium/high', '1=Low (0); 2=Medium (>0-0.13); 3=High (>0.13); NaN=BE data unmatched', 'Same 162 respondents missing.'),
 ('NatWalkInd_cat', 'NatWalkInd_very_low/low/high/very_high', '1=Very low (≤8); 2=Low (>8-10); 3=High (>10-13); 4=Very high (>13); NaN=BE data unmatched', 'Four categories, not three — there is no "medium" bucket for this index in the source file. Same 162 respondents missing.'),
]
for row in CAT_ROWS:
    for c, v in enumerate(row, start=1):
        cell = ws.cell(row=r, column=c, value=v); cell.font = normal; cell.border = border; cell.alignment = wrap
    r += 1

# ---------------- Event Sample Definitions ----------------
ws = wb.create_sheet('Event Sample Definitions')
style_header(ws, 1, ['Event','Filter column','N exposed','Activities modeled (9 possible)','WFH/WFO extra filter'], [16,18,10,45,30])
r=2
N = {}
for label, ev, flag in EVENTS:
    N[ev] = int(df[flag].sum())
ACT_LIST = {'heat':'Usual, Home, Car, Transit, WFH, WFO, Dine in, Pick up, Delivery (all 9)',
            'cold':'Usual, Home, Car, Transit, WFH, WFO, Dine in, Pick up, Delivery (all 9)',
            'powerout':'Usual, Home, Car, Transit, Dine in, Pick up, Delivery (no WFH/WFO — not asked)',
            'earthquake':'Usual, Home, Car, Transit, WFH (no WFO/Dine in/Pick up/Delivery — not asked)',
            'flooding':'Usual, Home, Car, Transit, WFH (no WFO/Dine in/Pick up/Delivery — not asked)'}
for label, ev, flag in EVENTS:
    row = [label, flag, N[ev], ACT_LIST[ev], 'empsta<3 (currently working) applied only to WFH/WFO models' if ev in ('heat','cold') else ('empsta<3 applied to WFH model' if ev in ('earthquake','flooding') else 'n/a — no WFH/WFO for this event')]
    for c, v in enumerate(row, start=1):
        cell = ws.cell(row=r, column=c, value=v); cell.font = normal; cell.border = border; cell.alignment = wrap
    r += 1

# ---------------- Data Quality Notes ----------------
ws = wb.create_sheet('Data Quality Notes')
ws.column_dimensions['A'].width = 100
r=1
ws.cell(row=r, column=1, value='Data Quality Notes').font = title_font
r+=2
notes = [
"1) hispanic — raw column is NOT a clean 0/1 dummy",
"   Raw coding: 1 = Hispanic (Yes), 2 = Not Hispanic (No), 0 = one ambiguous respondent.",
"   The original model-estimation code used this raw column directly as a regressor (candidate_vars",
"   included 'hispanic'), which several already-fitted final models kept as significant (e.g. WFH-Heat,",
"   WFO-Cold, Transit-Earthquake/Powerout). Because 2 > 1, the raw coefficient's sign is the OPPOSITE of",
"   the usual 'is Hispanic' dummy — e.g. WFH-Heat originally showed hispanic coef = -0.289; using the",
"   cleaned hispanic_c (1=Hispanic) the same relationship comes out as +0.249.",
"   FIX APPLIED: hispanic_c = 1 if raw==1, 0 if raw==2, NaN if raw==0. Use hispanic_c in all new models.",
"",
"2) ac — raw column mixes a real 0/1 dummy with a -9 skip-logic code",
"   Raw coding: 1 = has A/C, 0 = no A/C, -9 = 'Not shown' (question not routed to this respondent).",
"   The A/C question is asked mainly within the Heat module: only 1 of 2,653 heat-exposed respondents has",
"   ac=-9, but 55-70% of respondents exposed to the OTHER four events have ac=-9. The original code's",
"   candidate_vars included 'ac' for Cold, Flooding, Earthquake, and Power Outage models too, and .dropna()",
"   does not remove -9 (it is a real integer, not NaN) — so -9 got fit as if it were a legitimate numeric",
"   value. This shows up in the previously-published results as small but 'significant' ac coefficients in",
"   places that don't make substantive sense (e.g. Power Outage - Usual: coef=0.01, t=2.19; Earthquake -",
"   Home: coef=-0.018, t=-1.71; Flooding - Transit: coef=0.017, t=2.20) — each in samples where most",
"   respondents actually have ac=-9.",
"   FIX APPLIED: ac_c = NaN wherever raw ac==-9, otherwise unchanged. Use ac_c in all new models.",
"",
"3) CR / SE / PR are continuous, not dummies",
"   These are standardized factor scores (from factor_analyzer) summarizing community resilience, social",
"   engagement, and personal resilience attitude items, each with mean≈0 and SD≈1. They were already being",
"   used correctly as continuous regressors in the original code — flagged here only because the original",
"   codebook file didn't state this explicitly (it listed them with no value-coding, which could be misread",
"   as an undocumented dummy).",
]
for line in notes:
    ws.cell(row=r, column=1, value=line).font = bold if line and not line.startswith(' ') and line[0].isdigit() else normal
    r+=1

wb.save('./Codebook.xlsx')
print("saved Codebook.xlsx")


## Summary

Running this notebook produces, in the working directory:

- `event_data/heat_data.csv`, `cold_data.csv`, `powerout_data.csv`, `earthquake_data.csv`, `flooding_data.csv`
- `Model_Results.xlsx` — baseline + final ordered probit results for all 35 Event x Activity models
- `Codebook.xlsx` — full variable documentation, combined categorical variables, and data-quality notes

Two things worth double-checking against your original R workflow before treating any single
coefficient as final:
1. Any model that previously included `hispanic` or `ac` — the cleaned versions (`hispanic_c`,
   `ac_c`) can flip the sign or drop out of the final model relative to the previously published numbers.
2. Threshold/cutpoint comparisons against R — use the transformed thresholds this notebook reports,
   not raw `results.params`.